# Ricci Finance v10 Lecture

## v8 market topology + optional v9 Ricci-flow layer

This notebook is a graduate-level companion to the Streamlit app. The main design is:

```text
v8 sparse rolling topology  ->  selected frame  ->  optional v9 Ricci flow  ->  optional surgery
```

The reason for this design is practical: sparse threshold topology is more interpretable for market clusters and sector rotation, while Ricci flow/surgery is more useful as a research diagnostic for stress and singular bridges.

## 1. Setup

Put `helper.py` in the same directory as this notebook.

In [ ]:
import pandas as pd
import numpy as np

from helper import (
    DEFAULT_TICKERS, make_demo_prices, prices_to_returns, build_rolling_frames,
    build_base_graph_for_layout, compute_stable_layout, build_plotly_animation,
    visualize_network, summarize_edges, rolling_feature_table, plot_rolling_features,
    sector_flow_table, plot_sector_flow, run_ricci_flow, plot_ricci_flow_history,
    compare_before_after_flow, perform_financial_surgery, compute_components, compute_hmm_regimes
)

## 2. Synthetic market with IPO-aware tickers

QNT/BNT are treated as late-start tickers. They should not appear in the rolling graph until enough valid observations exist.

In [ ]:
tickers = DEFAULT_TICKERS + ["BNT"]
prices = make_demo_prices(tickers=tickers, n_days=320, seed=7, ipo_tickers=("QNT", "BNT"), ipo_start_frac=0.58)
returns = prices_to_returns(prices)
prices.tail()

## 3. Build v8-style sparse rolling topology

This is the default v10 market view. Use `edge_mode='threshold'` and positive correlation only to preserve clusters.

In [ ]:
frames, starts = build_rolling_frames(
    returns,
    window_size=60,
    step=5,
    max_frames=45,
    edge_mode="threshold",
    positive_only=True,
    max_distance=1.05,
    min_abs_corr=0.30,
    keep_top_edges=None,
    min_node_obs=1,
    min_pair_obs=4,
    alpha=0.5,
    method="OTD",
    proc=1,
)

feature_df = rolling_feature_table(frames)
feature_df.head()

## 4. Animate the raw topology

This is the v8 market-map layer. Edge labels show financial distance weight `w`: smaller values mean stronger positive correlation.

In [ ]:
base = build_base_graph_for_layout(frames, all_nodes=returns.columns)
pos = compute_stable_layout(base, seed=42)
fig = build_plotly_animation(
    frames, pos, title="Ricci Finance v10: raw v8 sparse topology",
    show_edge_weight_labels=True, edge_label_top_n=40
)
fig

## 5. Inspect a selected frame

This is the frame that the Streamlit app sends into the optional v9 layer.

In [ ]:
idx = len(frames) - 1
fd = frames[idx]
print(fd.stats)
visualize_network(fd.G, positions=pos, title="Selected raw v8 topology frame", node_cluster=fd.node_cluster)

In [ ]:
summarize_edges(fd.G).head(15)

## 6. Rolling diagnostics

Clusters, entropy, curvature and density reveal whether the market is fragmented or coherent.

In [ ]:
plot_rolling_features(feature_df)

## 7. Capital-flow proxy is separate from Ricci topology

Ricci topology is correlation geometry. Capital-flow proxy is recent return by theme. AI/Quantum can receive money but remain separate if their tickers do not co-move strongly in the window.

In [ ]:
flow_df = sector_flow_table(returns.iloc[: starts[idx] + 60], lookback=20)
flow_df

In [ ]:
plot_sector_flow(flow_df)

## 8. Optional v9 layer: Ricci flow

Ricci flow deforms edge distances. Positive curvature tends to contract coherent cluster links; negative curvature tends to stretch bridge/stress links.

In [ ]:
flowed_G, flow_history = run_ricci_flow(
    fd.G, iterations=10, step_size=0.25, alpha=0.5, method="OTD", proc=1, normalize_mean_weight=True
)
plot_ricci_flow_history(flow_history)

In [ ]:
compare_before_after_flow(fd.G, flowed_G).head(15)

## 9. Optional surgery

Financial surgery cuts singular edges: usually negative-curvature bridges or long unstable connections.

In [ ]:
surgery = perform_financial_surgery(
    flowed_G, curvature_threshold=-0.35, distance_quantile=0.80, use_bridge_test=True, remove_isolated_nodes=False
)
print("Removed edges:", surgery.removed_edges)
print("Before:", surgery.before_stats)
print("After:", surgery.after_stats)
surgery.report

In [ ]:
visualize_network(
    surgery.after, positions=pos, title="After Ricci flow + financial surgery",
    node_cluster=compute_components(surgery.after),
    highlight_edges=[(u, v) for u, v, _, _ in surgery.removed_edges],
)

## 10. HMM hidden regimes

HMM regimes summarize the rolling topology features. They are unsupervised labels, not trading signals by themselves.

In [ ]:
hmm_df, names = compute_hmm_regimes(frames, returns, starts, window_size=60, n_components=3, forward_days=5, random_state=42)
hmm_df.tail()

## 11. Lecture conclusion

- Use v8 topology for market interpretation, clusters, IPO emergence, and sector rotation.
- Use v9 Ricci flow/surgery as an optional stress-test layer.
- Do not confuse correlation geometry with capital flow.
- v10 is therefore a two-layer geometric finance system: market map first, research flow second.